In [1]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from array import array

# ROOT 설정
ROOT.gROOT.SetBatch(True)  # GUI 없이 실행
ROOT.gStyle.SetOptStat(0)  # 통계 박스 숨기기

def load_2d_histogram(file_path, hist_name, systematic="Central"):
    """
    2D 히스토그램을 ROOT 파일에서 로드하는 함수
    
    Parameters:
    - file_path: ROOT 파일 경로
    - hist_name: 히스토그램 이름
    - systematic: 시스템틱 디렉토리 (기본값: "Central")
    
    Returns:
    - 2D 히스토그램 객체 또는 None
    """
    root_file = ROOT.TFile.Open(file_path)
    if not root_file or root_file.IsZombie():
        print(f"파일을 열 수 없습니다: {file_path}")
        return None
    
    # 시스템틱 디렉토리로 이동
    directory = root_file.Get(systematic)
    if not directory:
        print(f"디렉토리를 찾을 수 없습니다: {systematic}")
        root_file.Close()
        return None
    
    # 2D 히스토그램 가져오기
    hist = directory.Get(hist_name)
    if not hist:
        print(f"히스토그램을 찾을 수 없습니다: {hist_name}")
        root_file.Close()
        return None
    
    # 히스토그램 복사 (파일이 닫혀도 사용 가능)
    hist_clone = hist.Clone(f"{os.path.basename(file_path)}_{hist_name}")
    hist_clone.SetDirectory(0)  # 메모리에 저장
    ROOT.SetOwnership(hist_clone, False)
    
    root_file.Close()
    return hist_clone

def analyze_2d_threshold_percentage(hist_2d, x_threshold, y_threshold):
    """
    2D 히스토그램에서 특정 x, y 임계값 이상의 영역이 전체의 몇 퍼센트를 차지하는지 계산
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_threshold: x축 임계값
    - y_threshold: y축 임계값
    
    Returns:
    - dict: 분석 결과 (퍼센트, 총 이벤트 수, 임계값 이상 이벤트 수 등)
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return None
    
    # 히스토그램 정보
    nbins_x = hist_2d.GetNbinsX()
    nbins_y = hist_2d.GetNbinsY()
    x_min = hist_2d.GetXaxis().GetXmin()
    x_max = hist_2d.GetXaxis().GetXmax()
    y_min = hist_2d.GetYaxis().GetXmin()
    y_max = hist_2d.GetYaxis().GetXmax()
    
    print(f"2D 히스토그램 정보:")
    print(f"  X축: {nbins_x}개 빈, 범위 [{x_min:.2f}, {x_max:.2f}]")
    print(f"  Y축: {nbins_y}개 빈, 범위 [{y_min:.2f}, {y_max:.2f}]")
    print(f"  요청된 임계값: x >= {x_threshold}, y >= {y_threshold}")
    
    # 총 이벤트 수 계산
    total_events = hist_2d.Integral()
    print(f"  총 이벤트 수: {total_events:.2f}")
    
    # 임계값 이상의 이벤트 수 계산
    threshold_events = 0.0
    
    for i in range(1, nbins_x + 1):
        for j in range(1, nbins_y + 1):
            bin_content = hist_2d.GetBinContent(i, j)
            if bin_content <= 0:
                continue
                
            # 빈의 중심 좌표 계산
            x_center = hist_2d.GetXaxis().GetBinCenter(i)
            y_center = hist_2d.GetYaxis().GetBinCenter(j)
            
            # 임계값 확인
            if x_center >= x_threshold and y_center >= y_threshold:
                threshold_events += bin_content
    
    # 퍼센트 계산
    percentage = (threshold_events / total_events) * 100 if total_events > 0 else 0
    
    result = {
        'total_events': total_events,
        'threshold_events': threshold_events,
        'percentage': percentage,
        'x_threshold': x_threshold,
        'y_threshold': y_threshold,
        'hist_info': {
            'nbins_x': nbins_x,
            'nbins_y': nbins_y,
            'x_range': [x_min, x_max],
            'y_range': [y_min, y_max]
        }
    }
    
    print(f"\n분석 결과:")
    print(f"  임계값 이상 이벤트 수: {threshold_events:.2f}")
    print(f"  전체 대비 비율: {percentage:.2f}%")
    
    return result

def visualize_2d_threshold(hist_2d, x_threshold, y_threshold, title="2D Histogram with Threshold"):
    """
    2D 히스토그램과 임계값을 시각화
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_threshold: x축 임계값
    - y_threshold: y축 임계값
    - title: 플롯 제목
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return
    
    # ROOT 캔버스 생성
    canvas = ROOT.TCanvas("canvas_2d", "2D Histogram", 800, 600)
    
    # 히스토그램 그리기
    hist_2d.Draw("COLZ")
    
    # 임계값 선 그리기
    # 수직선 (x = x_threshold)
    line_x = ROOT.TLine(x_threshold, hist_2d.GetYaxis().GetXmin(), 
                       x_threshold, hist_2d.GetYaxis().GetXmax())
    line_x.SetLineColor(ROOT.kRed)
    line_x.SetLineWidth(2)
    line_x.SetLineStyle(2)
    line_x.Draw()
    
    # 수평선 (y = y_threshold)
    line_y = ROOT.TLine(hist_2d.GetXaxis().GetXmin(), y_threshold,
                       hist_2d.GetXaxis().GetXmax(), y_threshold)
    line_y.SetLineColor(ROOT.kRed)
    line_y.SetLineWidth(2)
    line_y.SetLineStyle(2)
    line_y.Draw()
    
    # 제목 설정
    hist_2d.SetTitle(title)
    
    # 범례 추가
    legend = ROOT.TLegend(0.7, 0.8, 0.9, 0.9)
    legend.AddEntry(line_x, f"x >= {x_threshold}", "l")
    legend.AddEntry(line_y, f"y >= {y_threshold}", "l")
    legend.Draw()
    
    canvas.Update()
    return canvas

def analyze_multiple_thresholds(hist_2d, x_thresholds, y_thresholds):
    """
    여러 임계값에 대해 분석 수행
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_thresholds: x축 임계값 리스트
    - y_thresholds: y축 임계값 리스트
    
    Returns:
    - 결과 딕셔너리 리스트
    """
    results = []
    
    for x_thresh in x_thresholds:
        for y_thresh in y_thresholds:
            print(f"\n{'='*50}")
            print(f"임계값 분석: x >= {x_thresh}, y >= {y_thresh}")
            print(f"{'='*50}")
            
            result = analyze_2d_threshold_percentage(hist_2d, x_thresh, y_thresh)
            if result:
                results.append(result)
    
    return results

def print_analysis_summary(results):
    """
    분석 결과 요약 출력
    
    Parameters:
    - results: analyze_multiple_thresholds의 결과 리스트
    """
    if not results:
        print("분석 결과가 없습니다.")
        return
    
    print(f"\n{'='*80}")
    print("분석 결과 요약")
    print(f"{'='*80}")
    print(f"{'X 임계값':<12} {'Y 임계값':<12} {'이벤트 수':<15} {'비율 (%)':<12}")
    print(f"{'-'*80}")
    
    for result in results:
        print(f"{result['x_threshold']:<12.1f} {result['y_threshold']:<12.1f} "
              f"{result['threshold_events']:<15.2f} {result['percentage']:<12.2f}")

# 사용 예시 함수
def example_usage():
    """
    2D 히스토그램 분석 사용 예시
    """
    print("2D 히스토그램 분석 예시")
    print("="*50)
    
    # 예시: 데이터 디렉토리에서 2D 히스토그램 찾기
    data_directory = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel/2022"
    
    # ROOT 파일들 찾기
    root_files = glob.glob(os.path.join(data_directory, "*.root"))
    
    if not root_files:
        print(f"디렉토리에서 ROOT 파일을 찾을 수 없습니다: {data_directory}")
        return
    
    print(f"찾은 ROOT 파일 수: {len(root_files)}")
    
    # 첫 번째 파일에서 2D 히스토그램 찾기
    sample_file = root_files[0]
    print(f"샘플 파일: {os.path.basename(sample_file)}")
    
    # 가능한 2D 히스토그램 이름들 (실제 사용시 수정 필요)
    possible_2d_histograms = [
        "WRMass_vs_NMass",  # 예시
        "Mll_vs_Mjj",       # 예시
        "Pt_vs_Eta",        # 예시
    ]
    
    # 실제 사용시에는 적절한 2D 히스토그램 이름을 지정하세요
    hist_name = "WRMass_vs_NMass"  # 실제 히스토그램 이름으로 변경 필요
    
    # 2D 히스토그램 로드
    hist_2d = load_2d_histogram(sample_file, hist_name)
    
    if hist_2d:
        print(f"2D 히스토그램 '{hist_name}' 로드 성공!")
        
        # 임계값 설정 (실제 사용시 적절한 값으로 변경)
        x_thresholds = [1000, 2000, 3000]
        y_thresholds = [500, 1000, 1500]
        
        # 분석 수행
        results = analyze_multiple_thresholds(hist_2d, x_thresholds, y_thresholds)
        
        # 결과 요약 출력
        print_analysis_summary(results)
        
        # 시각화 (선택사항)
        if results:
            # 첫 번째 결과로 시각화
            first_result = results[0]
            canvas = visualize_2d_threshold(hist_2d, 
                                          first_result['x_threshold'], 
                                          first_result['y_threshold'],
                                          f"2D Histogram Analysis - {hist_name}")
            
            # 저장
            canvas.SaveAs("2d_threshold_analysis.png")
            print("시각화 결과가 '2d_threshold_analysis.png'로 저장되었습니다.")
            
    else:
        print(f"2D 히스토그램 '{hist_name}'을 찾을 수 없습니다.")
        print("사용 가능한 히스토그램을 확인하려면 ROOT 파일을 직접 열어보세요.")

#if __name__ == "__main__":
    # 예시 실행
#    example_usage()


In [2]:
# 실제 사용 예시 - 간단한 버전
def simple_2d_analysis(file_path, hist_name, x_threshold, y_threshold):
    """
    간단한 2D 히스토그램 분석 함수
    
    사용법:
    result = simple_2d_analysis("파일경로.root", "히스토그램이름", x_임계값, y_임계값)
    """
    
    print(f"2D 히스토그램 분석 시작")
    print(f"파일: {file_path}")
    print(f"히스토그램: {hist_name}")
    print(f"임계값: x >= {x_threshold}, y >= {y_threshold}")
    print("-" * 50)
    
    # 2D 히스토그램 로드
    hist_2d = load_2d_histogram(file_path, hist_name)
    
    if not hist_2d:
        print("히스토그램을 로드할 수 없습니다.")
        return None
    
    # 분석 수행
    result = analyze_2d_threshold_percentage(hist_2d, x_threshold, y_threshold)
    
    if result:
        print(f"\n🎯 최종 결과:")
        print(f"   x >= {x_threshold}, y >= {y_threshold} 영역이")
        print(f"   전체의 {result['percentage']:.2f}%를 차지합니다!")
        
        # 시각화
        canvas = visualize_2d_threshold(hist_2d, x_threshold, y_threshold, 
                                      f"2D Analysis: {hist_name}")
        canvas.SaveAs(f"2d_analysis_{x_threshold}_{y_threshold}.png")
        print(f"   시각화 결과가 '2d_analysis_{x_threshold}_{y_threshold}.png'로 저장되었습니다.")
    
    return result

# 사용 예시 (실제 파일 경로와 히스토그램 이름으로 변경하세요)
print("2D 히스토그램 분석 도구가 준비되었습니다!")
print("\n사용법:")
print("1. simple_2d_analysis('파일경로.root', '히스토그램이름', x_임계값, y_임계값)")
print("2. 또는 example_usage() 함수를 실행하여 자동으로 분석")
print("\n예시:")
print("result = simple_2d_analysis('/path/to/file.root', 'WRMass_vs_NMass', 2000, 1000)")


2D 히스토그램 분석 도구가 준비되었습니다!

사용법:
1. simple_2d_analysis('파일경로.root', '히스토그램이름', x_임계값, y_임계값)
2. 또는 example_usage() 함수를 실행하여 자동으로 분석

예시:
result = simple_2d_analysis('/path/to/file.root', 'WRMass_vs_NMass', 2000, 1000)


In [3]:
# 고급 분석 기능들
def find_optimal_thresholds(hist_2d, target_percentages=[10, 25, 50, 75, 90]):
    """
    특정 퍼센트에 해당하는 임계값을 찾는 함수
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - target_percentages: 찾고자 하는 퍼센트 리스트
    
    Returns:
    - 각 퍼센트에 해당하는 임계값 딕셔너리
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return None
    
    total_events = hist_2d.Integral()
    nbins_x = hist_2d.GetNbinsX()
    nbins_y = hist_2d.GetNbinsY()
    
    # 모든 빈의 내용을 수집하고 정렬
    bin_data = []
    for i in range(1, nbins_x + 1):
        for j in range(1, nbins_y + 1):
            bin_content = hist_2d.GetBinContent(i, j)
            if bin_content > 0:
                x_center = hist_2d.GetXaxis().GetBinCenter(i)
                y_center = hist_2d.GetYaxis().GetBinCenter(j)
                bin_data.append((bin_content, x_center, y_center))
    
    # 내용에 따라 내림차순 정렬
    bin_data.sort(key=lambda x: x[0], reverse=True)
    
    # 누적 합계 계산
    cumulative_sum = 0
    cumulative_percentage = 0
    
    results = {}
    
    for target_pct in target_percentages:
        target_events = total_events * target_pct / 100
        
        for content, x_center, y_center in bin_data:
            cumulative_sum += content
            cumulative_percentage = (cumulative_sum / total_events) * 100
            
            if cumulative_percentage >= target_pct:
                results[target_pct] = {
                    'x_threshold': x_center,
                    'y_threshold': y_center,
                    'events': cumulative_sum,
                    'percentage': cumulative_percentage
                }
                break
    
    return results

def create_threshold_contour_plot(hist_2d, x_range, y_range, n_points=20):
    """
    임계값에 따른 퍼센트 변화를 등고선으로 시각화
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_range: x축 범위 [min, max]
    - y_range: y축 범위 [min, max]
    - n_points: 각 축의 점 개수
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return None
    
    # 격자 생성
    x_values = np.linspace(x_range[0], x_range[1], n_points)
    y_values = np.linspace(y_range[0], y_range[1], n_points)
    
    # 퍼센트 매트릭스 생성
    percentage_matrix = np.zeros((n_points, n_points))
    
    for i, x_thresh in enumerate(x_values):
        for j, y_thresh in enumerate(y_values):
            result = analyze_2d_threshold_percentage(hist_2d, x_thresh, y_thresh)
            if result:
                percentage_matrix[j, i] = result['percentage']
    
    # matplotlib으로 등고선 플롯 생성
    plt.figure(figsize=(10, 8))
    contour = plt.contour(x_values, y_values, percentage_matrix, levels=20)
    plt.clabel(contour, inline=True, fontsize=8)
    plt.colorbar(label='Percentage (%)')
    plt.xlabel('X Threshold')
    plt.ylabel('Y Threshold')
    plt.title('Threshold vs Percentage Contour Plot')
    plt.grid(True, alpha=0.3)
    
    # 저장
    plt.savefig('threshold_contour_plot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return percentage_matrix

def batch_analyze_files(directory, hist_name, x_thresholds, y_thresholds):
    """
    여러 파일에 대해 배치 분석 수행
    
    Parameters:
    - directory: ROOT 파일들이 있는 디렉토리
    - hist_name: 분석할 히스토그램 이름
    - x_thresholds: x축 임계값 리스트
    - y_thresholds: y축 임계값 리스트
    """
    root_files = glob.glob(os.path.join(directory, "*.root"))
    
    if not root_files:
        print(f"디렉토리에서 ROOT 파일을 찾을 수 없습니다: {directory}")
        return []
    
    print(f"배치 분석 시작: {len(root_files)}개 파일")
    print("="*60)
    
    all_results = []
    
    for file_path in root_files:
        filename = os.path.basename(file_path)
        print(f"\n파일 분석 중: {filename}")
        print("-" * 40)
        
        hist_2d = load_2d_histogram(file_path, hist_name)
        
        if not hist_2d:
            print(f"  히스토그램 '{hist_name}'을 찾을 수 없습니다.")
            continue
        
        file_results = []
        
        for x_thresh in x_thresholds:
            for y_thresh in y_thresholds:
                result = analyze_2d_threshold_percentage(hist_2d, x_thresh, y_thresh)
                if result:
                    result['filename'] = filename
                    file_results.append(result)
        
        all_results.extend(file_results)
        print(f"  완료: {len(file_results)}개 분석 결과")
    
    # 결과 요약
    print(f"\n{'='*80}")
    print("배치 분석 결과 요약")
    print(f"{'='*80}")
    print(f"{'파일명':<20} {'X 임계값':<10} {'Y 임계값':<10} {'비율 (%)':<12}")
    print(f"{'-'*80}")
    
    for result in all_results:
        print(f"{result['filename']:<20} {result['x_threshold']:<10.1f} "
              f"{result['y_threshold']:<10.1f} {result['percentage']:<12.2f}")
    
    return all_results

# 사용 예시 함수들
def quick_analysis_example():
    """
    빠른 분석 예시
    """
    print("🚀 빠른 2D 분석 예시")
    print("="*50)
    
    # 실제 사용시 이 값들을 수정하세요
    file_path = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel/2022/TBChannel_WR3000N1300.root"
    hist_name = "WRMass_vs_NMass"  # 실제 히스토그램 이름으로 변경
    x_threshold = 2000
    y_threshold = 1000
    
    print(f"분석할 파일: {file_path}")
    print(f"히스토그램: {hist_name}")
    print(f"임계값: x >= {x_threshold}, y >= {y_threshold}")
    
    # 분석 실행
    result = simple_2d_analysis(file_path, hist_name, x_threshold, y_threshold)
    
    return result

print("고급 분석 기능들이 추가되었습니다!")
print("\n사용 가능한 함수들:")
print("1. simple_2d_analysis() - 기본 분석")
print("2. find_optimal_thresholds() - 최적 임계값 찾기")
print("3. create_threshold_contour_plot() - 등고선 플롯")
print("4. batch_analyze_files() - 배치 분석")
print("5. quick_analysis_example() - 빠른 예시")


고급 분석 기능들이 추가되었습니다!

사용 가능한 함수들:
1. simple_2d_analysis() - 기본 분석
2. find_optimal_thresholds() - 최적 임계값 찾기
3. create_threshold_contour_plot() - 등고선 플롯
4. batch_analyze_files() - 배치 분석
5. quick_analysis_example() - 빠른 예시


In [4]:
# 실제 사용을 위한 테스트 코드
def test_with_available_files():
    """
    사용 가능한 파일들로 테스트 실행
    """
    print("🔍 사용 가능한 파일들로 테스트 실행")
    print("="*60)
    
    # 데이터 디렉토리들 확인
    possible_directories = [
        "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel/2022",
        "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel/2022EE",
        "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022",
        "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE"
    ]
    
    available_files = []
    
    for directory in possible_directories:
        if os.path.exists(directory):
            root_files = glob.glob(os.path.join(directory, "*.root"))
            if root_files:
                print(f"✅ 디렉토리 발견: {directory}")
                print(f"   파일 수: {len(root_files)}")
                available_files.extend(root_files[:3])  # 처음 3개 파일만
            else:
                print(f"❌ 디렉토리 비어있음: {directory}")
        else:
            print(f"❌ 디렉토리 없음: {directory}")
    
    if not available_files:
        print("\n⚠️ 사용 가능한 ROOT 파일을 찾을 수 없습니다.")
        print("파일 경로를 확인해주세요.")
        return None
    
    print(f"\n📁 총 {len(available_files)}개 파일 발견")
    
    # 첫 번째 파일로 테스트
    test_file = available_files[0]
    print(f"\n🧪 테스트 파일: {os.path.basename(test_file)}")
    
    # 가능한 2D 히스토그램 이름들 (실제로는 파일을 열어서 확인해야 함)
    possible_histograms = [
        "WRMass_vs_NMass",
        "Mll_vs_Mjj", 
        "Pt_vs_Eta",
        "Eta_vs_Phi",
        "Mass_vs_Pt"
    ]
    
    print(f"\n🔍 가능한 2D 히스토그램들:")
    for hist_name in possible_histograms:
        print(f"  - {hist_name}")
    
    print(f"\n💡 사용법:")
    print(f"1. 실제 히스토그램 이름을 확인한 후:")
    print(f"   result = simple_2d_analysis('{test_file}', '실제히스토그램이름', 2000, 1000)")
    print(f"\n2. 또는 여러 임계값으로 분석:")
    print(f"   hist_2d = load_2d_histogram('{test_file}', '실제히스토그램이름')")
    print(f"   results = analyze_multiple_thresholds(hist_2d, [1000, 2000, 3000], [500, 1000, 1500])")
    
    return test_file

# 실행 예시
print("테스트 함수가 준비되었습니다!")
print("실행하려면: test_with_available_files()")


테스트 함수가 준비되었습니다!
실행하려면: test_with_available_files()


# 2D 히스토그램 임계값 분석 도구

이 노트북은 ROOT 파일의 2D 히스토그램에서 특정 x, y 값 이상이 전체의 몇 퍼센트를 차지하는지 분석하는 도구입니다.

## 🚀 빠른 시작

### 1. 기본 사용법
```python
# 단일 임계값 분석
result = simple_2d_analysis("파일경로.root", "히스토그램이름", x_임계값, y_임계값)
```

### 2. 여러 임계값 분석
```python
# 히스토그램 로드
hist_2d = load_2d_histogram("파일경로.root", "히스토그램이름")

# 여러 임계값으로 분석
results = analyze_multiple_thresholds(hist_2d, [1000, 2000, 3000], [500, 1000, 1500])
```

### 3. 배치 분석
```python
# 여러 파일에 대해 분석
results = batch_analyze_files("디렉토리경로", "히스토그램이름", [1000, 2000], [500, 1000])
```

## 📊 주요 기능

1. **기본 분석**: 특정 임계값에서의 퍼센트 계산
2. **시각화**: 임계값이 표시된 2D 플롯 생성
3. **최적 임계값 찾기**: 특정 퍼센트에 해당하는 임계값 자동 탐색
4. **등고선 플롯**: 임계값에 따른 퍼센트 변화 시각화
5. **배치 처리**: 여러 파일에 대한 일괄 분석

## 🔧 사용 가능한 함수들

- `load_2d_histogram()`: 2D 히스토그램 로드
- `analyze_2d_threshold_percentage()`: 임계값 분석
- `simple_2d_analysis()`: 간단한 분석 (추천)
- `visualize_2d_threshold()`: 시각화
- `find_optimal_thresholds()`: 최적 임계값 찾기
- `create_threshold_contour_plot()`: 등고선 플롯
- `batch_analyze_files()`: 배치 분석
- `test_with_available_files()`: 테스트 실행

## 💡 사용 예시

```python
# 1. 테스트 실행
test_with_available_files()

# 2. 실제 분석
result = simple_2d_analysis("/path/to/file.root", "WRMass_vs_NMass", 2000, 1000)
print(f"x >= 2000, y >= 1000 영역이 전체의 {result['percentage']:.2f}%를 차지합니다!")
```


In [5]:
# 4개 영역 분석 함수들
def analyze_4_regions(hist_2d, x_threshold, y_threshold):
    """
    2D 히스토그램을 x, y 임계값을 기준으로 4개 영역으로 나누어 각 영역의 비율을 계산
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_threshold: x축 임계값
    - y_threshold: y축 임계값
    
    Returns:
    - dict: 4개 영역의 분석 결과
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return None
    
    # 히스토그램 정보
    nbins_x = hist_2d.GetNbinsX()
    nbins_y = hist_2d.GetNbinsY()
    x_min = hist_2d.GetXaxis().GetXmin()
    x_max = hist_2d.GetXaxis().GetXmax()
    y_min = hist_2d.GetYaxis().GetXmin()
    y_max = hist_2d.GetYaxis().GetXmax()
    
    print(f"2D 히스토그램 4개 영역 분석")
    print(f"  X축: {nbins_x}개 빈, 범위 [{x_min:.2f}, {x_max:.2f}]")
    print(f"  Y축: {nbins_y}개 빈, 범위 [{y_min:.2f}, {y_max:.2f}]")
    print(f"  분할 기준: x = {x_threshold}, y = {y_threshold}")
    print(f"  {'='*60}")
    
    # 총 이벤트 수 계산
    total_events = hist_2d.Integral()
    print(f"  총 이벤트 수: {total_events:.2f}")
    
    # 4개 영역별 이벤트 수 계산
    regions = {
        'region_1': {'name': 'x < threshold, y < threshold', 'events': 0.0, 'x_condition': '<', 'y_condition': '<'},
        'region_2': {'name': 'x >= threshold, y < threshold', 'events': 0.0, 'x_condition': '>=', 'y_condition': '<'},
        'region_3': {'name': 'x < threshold, y >= threshold', 'events': 0.0, 'x_condition': '<', 'y_condition': '>='},
        'region_4': {'name': 'x >= threshold, y >= threshold', 'events': 0.0, 'x_condition': '>=', 'y_condition': '>='}
    }
    
    for i in range(1, nbins_x + 1):
        for j in range(1, nbins_y + 1):
            bin_content = hist_2d.GetBinContent(i, j)
            if bin_content <= 0:
                continue
                
            # 빈의 중심 좌표 계산
            x_center = hist_2d.GetXaxis().GetBinCenter(i)
            y_center = hist_2d.GetYaxis().GetBinCenter(j)
            
            # 4개 영역 분류
            if x_center < x_threshold and y_center < y_threshold:
                regions['region_1']['events'] += bin_content
            elif x_center >= x_threshold and y_center < y_threshold:
                regions['region_2']['events'] += bin_content
            elif x_center < x_threshold and y_center >= y_threshold:
                regions['region_3']['events'] += bin_content
            elif x_center >= x_threshold and y_center >= y_threshold:
                regions['region_4']['events'] += bin_content
    
    # 퍼센트 계산 및 결과 정리
    results = {}
    print(f"\n  📊 4개 영역 분석 결과:")
    print(f"  {'-'*60}")
    
    for region_key, region_data in regions.items():
        percentage = (region_data['events'] / total_events) * 100 if total_events > 0 else 0
        region_data['percentage'] = percentage
        region_data['x_threshold'] = x_threshold
        region_data['y_threshold'] = y_threshold
        
        print(f"  {region_data['name']:<35}: {region_data['events']:>8.2f} ({percentage:>6.2f}%)")
        
        results[region_key] = region_data
    
    # 검증: 모든 영역의 합이 총 이벤트와 일치하는지 확인
    total_region_events = sum(region['events'] for region in regions.values())
    print(f"  {'-'*60}")
    print(f"  검증 - 영역별 합계: {total_region_events:.2f} (총 이벤트: {total_events:.2f})")
    print(f"  차이: {abs(total_region_events - total_events):.6f}")
    
    return results

def visualize_4_regions(hist_2d, x_threshold, y_threshold, title="2D Histogram - 4 Regions Analysis"):
    """
    4개 영역을 시각화하는 함수
    
    Parameters:
    - hist_2d: 2D 히스토그램 객체
    - x_threshold: x축 임계값
    - y_threshold: y축 임계값
    - title: 플롯 제목
    """
    if not hist_2d:
        print("유효하지 않은 히스토그램입니다.")
        return None
    
    # ROOT 캔버스 생성
    canvas = ROOT.TCanvas("canvas_4regions", "4 Regions Analysis", 1000, 800)
    
    # 히스토그램 그리기
    hist_2d.Draw("COLZ")
    
    # 임계값 선 그리기
    # 수직선 (x = x_threshold)
    line_x = ROOT.TLine(x_threshold, hist_2d.GetYaxis().GetXmin(), 
                       x_threshold, hist_2d.GetYaxis().GetXmax())
    line_x.SetLineColor(ROOT.kRed)
    line_x.SetLineWidth(3)
    line_x.SetLineStyle(1)
    line_x.Draw()
    
    # 수평선 (y = y_threshold)
    line_y = ROOT.TLine(hist_2d.GetXaxis().GetXmin(), y_threshold,
                       hist_2d.GetXaxis().GetXmax(), y_threshold)
    line_y.SetLineColor(ROOT.kRed)
    line_y.SetLineWidth(3)
    line_y.SetLineStyle(1)
    line_y.Draw()
    
    # 영역별 텍스트 라벨 추가
    x_min = hist_2d.GetXaxis().GetXmin()
    x_max = hist_2d.GetXaxis().GetXmax()
    y_min = hist_2d.GetYaxis().GetXmin()
    y_max = hist_2d.GetYaxis().GetXmax()
    
    # 영역 1: x < threshold, y < threshold (좌하단)
    text1 = ROOT.TLatex((x_min + x_threshold) / 2, (y_min + y_threshold) / 2, "Region 1")
    text1.SetTextColor(ROOT.kWhite)
    text1.SetTextSize(0.03)
    text1.SetTextAlign(22)
    text1.Draw()
    
    # 영역 2: x >= threshold, y < threshold (우하단)
    text2 = ROOT.TLatex((x_threshold + x_max) / 2, (y_min + y_threshold) / 2, "Region 2")
    text2.SetTextColor(ROOT.kWhite)
    text2.SetTextSize(0.03)
    text2.SetTextAlign(22)
    text2.Draw()
    
    # 영역 3: x < threshold, y >= threshold (좌상단)
    text3 = ROOT.TLatex((x_min + x_threshold) / 2, (y_threshold + y_max) / 2, "Region 3")
    text3.SetTextColor(ROOT.kWhite)
    text3.SetTextSize(0.03)
    text3.SetTextAlign(22)
    text3.Draw()
    
    # 영역 4: x >= threshold, y >= threshold (우상단)
    text4 = ROOT.TLatex((x_threshold + x_max) / 2, (y_threshold + y_max) / 2, "Region 4")
    text4.SetTextColor(ROOT.kWhite)
    text4.SetTextSize(0.03)
    text4.SetTextAlign(22)
    text4.Draw()
    
    # 제목 설정
    hist_2d.SetTitle(title)
    
    # 범례 추가
    legend = ROOT.TLegend(0.15, 0.85, 0.35, 0.95)
    legend.AddEntry(line_x, f"x = {x_threshold}", "l")
    legend.AddEntry(line_y, f"y = {y_threshold}", "l")
    legend.SetTextSize(0.03)
    legend.Draw()
    
    canvas.Update()
    return canvas

def simple_4regions_analysis(file_path, hist_name, x_threshold, y_threshold):
    """
    4개 영역 분석을 위한 간단한 함수
    
    Parameters:
    - file_path: ROOT 파일 경로
    - hist_name: 히스토그램 이름
    - x_threshold: x축 임계값
    - y_threshold: y축 임계값
    """
    print(f"🔍 4개 영역 분석 시작")
    print(f"파일: {file_path}")
    print(f"히스토그램: {hist_name}")
    print(f"분할 기준: x = {x_threshold}, y = {y_threshold}")
    print("="*60)
    
    # 2D 히스토그램 로드
    hist_2d = load_2d_histogram(file_path, hist_name)
    
    if not hist_2d:
        print("히스토그램을 로드할 수 없습니다.")
        return None
    
    # 4개 영역 분석 수행
    results = analyze_4_regions(hist_2d, x_threshold, y_threshold)
    
    if results:
        print(f"\n🎯 최종 결과 요약:")
        print(f"   영역 1 (x < {x_threshold}, y < {y_threshold}): {results['region_1']['percentage']:.2f}%")
        print(f"   영역 2 (x >= {x_threshold}, y < {y_threshold}): {results['region_2']['percentage']:.2f}%")
        print(f"   영역 3 (x < {x_threshold}, y >= {y_threshold}): {results['region_3']['percentage']:.2f}%")
        print(f"   영역 4 (x >= {x_threshold}, y >= {y_threshold}): {results['region_4']['percentage']:.2f}%")
        
        # 시각화
        canvas = visualize_4_regions(hist_2d, x_threshold, y_threshold, 
                                   f"4 Regions Analysis: {hist_name}")
        canvas.SaveAs(f"4regions_analysis_{x_threshold}_{y_threshold}.png")
        print(f"\n   시각화 결과가 '4regions_analysis_{x_threshold}_{y_threshold}.png'로 저장되었습니다.")
    
    return results

print("4개 영역 분석 기능이 추가되었습니다!")
print("\n사용법:")
print("result = simple_4regions_analysis('파일경로.root', '히스토그램이름', x_임계값, y_임계값)")
print("\n예시:")
print("result = simple_4regions_analysis('/path/to/file.root', 'mt vs subleading lepton ptCR', 200, 50)")


4개 영역 분석 기능이 추가되었습니다!

사용법:
result = simple_4regions_analysis('파일경로.root', '히스토그램이름', x_임계값, y_임계값)

예시:
result = simple_4regions_analysis('/path/to/file.root', 'mt vs subleading lepton ptCR', 200, 50)


In [6]:
# 4개 영역 분석 테스트
print("🧪 4개 영역 분석 테스트 실행")
print("="*60)

# 이전에 사용한 파일과 히스토그램으로 테스트
file_path = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TTLJ_powheg.root"
hist_name = "mt vs subleading lepton ptCR"
x_threshold = 200
y_threshold = 50

print(f"테스트 파일: {file_path}")
print(f"히스토그램: {hist_name}")
print(f"분할 기준: x = {x_threshold}, y = {y_threshold}")
print()

# 4개 영역 분석 실행
result = simple_4regions_analysis(file_path, hist_name, x_threshold, y_threshold)

if result:
    print(f"\n📈 상세 결과:")
    for region_key, region_data in result.items():
        print(f"  {region_data['name']}: {region_data['events']:.2f} 이벤트 ({region_data['percentage']:.2f}%)")


🧪 4개 영역 분석 테스트 실행
테스트 파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TTLJ_powheg.root
히스토그램: mt vs subleading lepton ptCR
분할 기준: x = 200, y = 50

🔍 4개 영역 분석 시작
파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TTLJ_powheg.root
히스토그램: mt vs subleading lepton ptCR
분할 기준: x = 200, y = 50
히스토그램을 찾을 수 없습니다: mt vs subleading lepton ptCR
히스토그램을 로드할 수 없습니다.


In [7]:
# 시그널 파일로도 4개 영역 분석 테스트
print("\n" + "="*60)
print("🚀 시그널 파일 4개 영역 분석")
print("="*60)

# 시그널 파일로 테스트
signal_file = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TBChannel_WR3000N1300.root"
hist_name = "mt vs subleading lepton ptCR"
x_threshold = 200
y_threshold = 50

print(f"시그널 파일: {signal_file}")
print(f"히스토그램: {hist_name}")
print(f"분할 기준: x = {x_threshold}, y = {y_threshold}")
print()

# 4개 영역 분석 실행
signal_result = simple_4regions_analysis(signal_file, hist_name, x_threshold, y_threshold)

if signal_result:
    print(f"\n📈 시그널 상세 결과:")
    for region_key, region_data in signal_result.items():
        print(f"  {region_data['name']}: {region_data['events']:.2f} 이벤트 ({region_data['percentage']:.2f}%)")
    
    # 배경과 시그널 비교
    print(f"\n🔄 배경 vs 시그널 비교:")
    print(f"  {'영역':<25} {'배경 (%)':<12} {'시그널 (%)':<12} {'차이 (%)':<12}")
    print(f"  {'-'*65}")
    
    if result and signal_result:
        for region_key in result.keys():
            bg_pct = result[region_key]['percentage']
            sig_pct = signal_result[region_key]['percentage']
            diff = sig_pct - bg_pct
            region_name = result[region_key]['name'][:20] + "..."
            print(f"  {region_name:<25} {bg_pct:<12.2f} {sig_pct:<12.2f} {diff:<12.2f}")



🚀 시그널 파일 4개 영역 분석
시그널 파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TBChannel_WR3000N1300.root
히스토그램: mt vs subleading lepton ptCR
분할 기준: x = 200, y = 50

🔍 4개 영역 분석 시작
파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TBChannel_WR3000N1300.root
히스토그램: mt vs subleading lepton ptCR
분할 기준: x = 200, y = 50
히스토그램을 찾을 수 없습니다: mt vs subleading lepton ptCR
히스토그램을 로드할 수 없습니다.


# 4개 영역 분석 기능 추가! 🎯

이제 x, y 경계를 기준으로 4개 영역의 비율을 구할 수 있습니다!

## 📊 4개 영역 분할

```
        y >= threshold
        ┌─────────────┬─────────────┐
        │   Region 3  │   Region 4  │
        │ x < th,     │ x >= th,    │
        │ y >= th     │ y >= th     │
        ├─────────────┼─────────────┤
        │   Region 1  │   Region 2  │
        │ x < th,     │ x >= th,    │
        │ y < th      │ y < th      │
        └─────────────┴─────────────┘
        x < threshold    x >= threshold
```

## 🚀 사용법

### 기본 사용법
```python
# 4개 영역 분석
result = simple_4regions_analysis("파일경로.root", "히스토그램이름", x_임계값, y_임계값)
```

### 결과 해석
- **Region 1**: x < 임계값, y < 임계값 (좌하단)
- **Region 2**: x >= 임계값, y < 임계값 (우하단)  
- **Region 3**: x < 임계값, y >= 임계값 (좌상단)
- **Region 4**: x >= 임계값, y >= 임계값 (우상단)

### 시각화
각 영역이 명확히 표시된 2D 플롯이 자동으로 생성됩니다!

## 💡 활용 예시

```python
# 배경과 시그널 비교
bg_result = simple_4regions_analysis("background.root", "hist_name", 200, 50)
sig_result = simple_4regions_analysis("signal.root", "hist_name", 200, 50)

# 각 영역별로 배경과 시그널의 분포 차이를 분석할 수 있습니다!
```


In [10]:
signal_file = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TBChannel_WR3000N1300.root"
hist_name = "mt vs subleading lepton ptall_WR"
x_threshold = 200
y_threshold = 50
signal_result = simple_4regions_analysis(signal_file, hist_name, x_threshold, y_threshold)

🔍 4개 영역 분석 시작
파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TBChannel_WR3000N1300.root
히스토그램: mt vs subleading lepton ptall_WR
분할 기준: x = 200, y = 50
2D 히스토그램 4개 영역 분석
  X축: 300개 빈, 범위 [0.00, 3000.00]
  Y축: 400개 빈, 범위 [0.00, 4000.00]
  분할 기준: x = 200, y = 50
  총 이벤트 수: 11.95

  📊 4개 영역 분석 결과:
  ------------------------------------------------------------
  x < threshold, y < threshold       :     0.01 (  0.05%)
  x >= threshold, y < threshold      :     0.21 (  1.76%)
  x < threshold, y >= threshold      :     0.32 (  2.71%)
  x >= threshold, y >= threshold     :    11.41 ( 95.49%)
  ------------------------------------------------------------
  검증 - 영역별 합계: 11.95 (총 이벤트: 11.95)
  차이: 0.000000

🎯 최종 결과 요약:
   영역 1 (x < 200, y < 50): 0.05%
   영역 2 (x >= 200, y < 50): 1.76%
   영역 3 (x < 200, y >= 50): 2.71%
   영역 4 (x >= 200, y >= 50): 95.49%

   시각화 결과가 '4regions_analysis_200_50.png'로 저장되었습니다.


Info in <TCanvas::Print>: png file 4regions_analysis_200_50.png has been created


In [11]:
signal_file = "/gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TTLJ_powheg.root"

In [12]:
signal_result = simple_4regions_analysis(signal_file, hist_name, x_threshold, y_threshold)

🔍 4개 영역 분석 시작
파일: /gv0/Users/achihwan/SKNanoRunlog/out/LRSM_TBChannel_notusingbjet/2022EE/TTLJ_powheg.root
히스토그램: mt vs subleading lepton ptall_WR
분할 기준: x = 200, y = 50
2D 히스토그램 4개 영역 분석
  X축: 300개 빈, 범위 [0.00, 3000.00]
  Y축: 400개 빈, 범위 [0.00, 4000.00]
  분할 기준: x = 200, y = 50
  총 이벤트 수: 518.59

  📊 4개 영역 분석 결과:
  ------------------------------------------------------------
  x < threshold, y < threshold       :   139.75 ( 26.95%)
  x >= threshold, y < threshold      :   134.49 ( 25.93%)
  x < threshold, y >= threshold      :   106.03 ( 20.45%)
  x >= threshold, y >= threshold     :   138.68 ( 26.74%)
  ------------------------------------------------------------
  검증 - 영역별 합계: 518.95 (총 이벤트: 518.59)
  차이: 0.356736

🎯 최종 결과 요약:
   영역 1 (x < 200, y < 50): 26.95%
   영역 2 (x >= 200, y < 50): 25.93%
   영역 3 (x < 200, y >= 50): 20.45%
   영역 4 (x >= 200, y >= 50): 26.74%

   시각화 결과가 '4regions_analysis_200_50.png'로 저장되었습니다.


Info in <TCanvas::Print>: png file 4regions_analysis_200_50.png has been created
